# Transposed Convolution and Upsampling Foundations

Autoencoder decoders need mechanisms that increase spatial resolution. This notebook builds the main ideas from first principles using NumPy:

- standard 2D convolution,
- transposed convolution,
- output-size behavior,
- and nearest-neighbor-style upsampling with repeated rows and columns.

These operations provide the intuition behind decoder architectures used later in convolutional autoencoders.


## Create a Small Example Image


In [1]:
import numpy as np
np.random.seed(0)
img =  np.random.randint(0,10,size = (6,6))

print(img)


[[5 0 3 3 7 9]
 [3 5 2 4 7 6]
 [8 8 1 6 7 7]
 [8 1 5 9 8 9]
 [4 3 0 3 5 0]
 [2 3 8 1 3 3]]


## Standard 2D Convolution

The function below implements a valid 2D convolution-style operation by sliding a kernel over the input array and computing the sum of element-wise products.


In [2]:
def convolve_2d(img, kernel):
    output = np.zeros((img.shape[0]-kernel.shape[0]+1, img.shape[1]-kernel.shape[1]+1),dtype = img.dtype)
    for i in range(output.shape[0]):
        for j in range(output.shape[1]):
            # Extract the subsection of the array
            sub_img = img[i:i + kernel.shape[0], j:j + kernel.shape[1]]
            # Perform element-wise multiplication and sum
            output[i, j] = np.sum(sub_img * kernel)
    return output


## Convolution Example


In [3]:
np.random.seed(2)
img =  np.random.randint(0,10,size = (4,6))
kernel =  np.array([[1,2],[2,1]])

print("img:\n", img)
print("kernel:\n", kernel)
print("convolve_2d(img, kernel):\n", convolve_2d(img, kernel))


img:
 [[8 8 6 2 8 7]
 [2 1 5 4 4 5]
 [7 3 6 4 3 7]
 [6 1 3 5 8 4]]
kernel:
 [[1 2]
 [2 1]]
convolve_2d(img, kernel):
 [[29 27 24 30 35]
 [21 23 29 23 27]
 [26 20 25 28 37]]


## Transposed Convolution from Scratch

A transposed convolution distributes each input value over an output region using the kernel. The contributions from overlapping regions are accumulated.


In [4]:
def convolve_2d_transpose(img, kernel):
    output = np.zeros((img.shape[0]+kernel.shape[0]-1, img.shape[1]+kernel.shape[1]-1),dtype = img.dtype)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            output[i:i + kernel.shape[0], j:j +kernel.shape[1]] += img[i,j]*kernel
    return output


## Transposed Convolution Example 1


In [5]:
np.random.seed(0)
img =  np.random.randint(0,10,size = (4,7))
kernel =  np.array([[1,2],[2,1]])

print("img:\n", img)
print("kernel:\n", kernel)
print("convolve_2d_transpose(img, kernel):\n", convolve_2d_transpose(img, kernel))


img:
 [[5 0 3 3 7 9 3]
 [5 2 4 7 6 8 8]
 [1 6 7 7 8 1 5]
 [9 8 9 4 3 0 3]]
kernel:
 [[1 2]
 [2 1]]
convolve_2d_transpose(img, kernel):
 [[ 5 10  3  9 13 23 21  6]
 [15 17 14 24 37 45 39 19]
 [11 17 29 39 41 39 31 18]
 [11 39 45 43 34 16 14 11]
 [18 25 26 17 10  3  6  3]]


## Transposed Convolution Example 2


In [6]:
np.random.seed(0)
img =  np.random.randint(0,6,size = (3,4))
kernel =  np.array([[1,2],[2,1]])

print("img:\n", img)
print("kernel:\n", kernel)
print("convolve_2d_transpose(img, kernel):\n", convolve_2d_transpose(img, kernel))


img:
 [[4 5 0 3]
 [3 3 1 3]
 [5 2 4 0]]
kernel:
 [[1 2]
 [2 1]]
convolve_2d_transpose(img, kernel):
 [[ 4 13 10  3  6]
 [11 23 12 11  9]
 [11 21 13 15  3]
 [10  9 10  4  0]]


## Explicit 2D Upsampling

A simpler alternative to learned transposed convolution is deterministic upsampling. The function below repeats rows and columns independently.


In [7]:
def upsample2D(img, sr=2, sc=2):
    img = np.repeat(img, sr, axis=0)
    return np.repeat(img, sc, axis=1)

np.random.seed(0)
img =  np.random.randint(0,10,size = (3,4))

print("img:\n", img)
print("upsample2D(img):\n", upsample2D(img))
print("upsample2D(img,1,2):\n", upsample2D(img,1,2))
print("upsample2D(img,2,3):\n", upsample2D(img,2,3))


img:
 [[5 0 3 3]
 [7 9 3 5]
 [2 4 7 6]]
upsample2D(img):
 [[5 5 0 0 3 3 3 3]
 [5 5 0 0 3 3 3 3]
 [7 7 9 9 3 3 5 5]
 [7 7 9 9 3 3 5 5]
 [2 2 4 4 7 7 6 6]
 [2 2 4 4 7 7 6 6]]
upsample2D(img,1,2):
 [[5 5 0 0 3 3 3 3]
 [7 7 9 9 3 3 5 5]
 [2 2 4 4 7 7 6 6]]
upsample2D(img,2,3):
 [[5 5 5 0 0 0 3 3 3 3 3 3]
 [5 5 5 0 0 0 3 3 3 3 3 3]
 [7 7 7 9 9 9 3 3 3 5 5 5]
 [7 7 7 9 9 9 3 3 3 5 5 5]
 [2 2 2 4 4 4 7 7 7 6 6 6]
 [2 2 2 4 4 4 7 7 7 6 6 6]]


## Key Takeaways

Standard convolution reduces or preserves spatial structure depending on padding and stride. Transposed convolution can learn how to increase spatial resolution, while explicit upsampling increases resolution deterministically and is often followed by a convolutional layer.

These decoder operations are used directly in convolutional autoencoders and encoder-decoder architectures.
